# Modal Prediction with Masked Language Modeling

This notebook evaluates whether a masked language model can recover the modal verb in modal + infinitive constructions.

The modal verb is masked in sentence context, and the model is evaluated on its ability to predict one of the three target modal lemmas: *dovere*, *potere*, and *volere*.

The notebook expects as input:
- Sketch Engine KWIC exports for the three modal verbs;
- covarying collexeme output from the collostructional analysis.

## Setup

In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers numpy pandas rbo tqdm scikit-learn

In [ ]:
import os
import re
import math
import random
import io
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM

## Configuration

In [ ]:
MODEL_NAME = "indigo-ai/BERTino"
DEVICE = "cuda"
SEED = 42

KWIC_FILES = {
    "dovere": "dovere_raw_kwic.csv",
    "potere": "potere_raw_kwic.csv",
    "volere": "volere_raw_kwic.csv",
}

COVAR_FILE = "covar_out.csv"

OUT_DIR = "out_task2_modal_mask"
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available() and DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("Configuration loaded.")

## Reading and balancing KWIC exports

The three modal datasets are loaded from Sketch Engine KWIC exports and balanced by sampling the same number of concordance lines for each modal.

In [ ]:
SUBSAMPLE_PER_MODAL = 9500

def parse_sketch_kwic(path):
    rows = []
    path = Path(path)

    def unwrap_and_parse(cell):
        if cell.startswith('"') and cell.endswith('"'):
            cell_inner = cell[1:-1].replace('""', '"')
        else:
            cell_inner = cell

        return next(
            csv.reader(
                io.StringIO(cell_inner),
                delimiter=",",
                quotechar='"',
                doublequote=True
            )
        )

    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        header_line = f.readline().rstrip("\n")
        first_field = header_line.split(";", 1)[0]
        header_fields = unwrap_and_parse(first_field)

        if len(header_fields) != 4:
            raise ValueError(f"Unexpected header in {path}: {header_fields}")

        for line in f:
            first = line.rstrip("\n").split(";", 1)[0]

            try:
                ref, left, kwic, right = unwrap_and_parse(first)
            except Exception:
                continue

            rows.append((ref, left, kwic, right))

    df = pd.DataFrame(rows, columns=["Reference", "Left", "KWIC", "Right"])

    return df


parts = []

for modal, path in KWIC_FILES.items():
    df = parse_sketch_kwic(path)

    print(f"{path}: loaded {len(df)} rows")

    if len(df) < SUBSAMPLE_PER_MODAL:
        print(f"Warning: {path} has only {len(df)} rows; keeping all rows.")
        ss = df.copy()
    else:
        ss = df.sample(n=SUBSAMPLE_PER_MODAL, random_state=SEED).copy()

    ss["modal_lemma"] = modal
    ss["src_file"] = path

    parts.append(ss)

data = pd.concat(parts, ignore_index=True)

print("Merged dataset size:", len(data))
print("Columns:", list(data.columns))

data.to_csv(os.path.join(OUT_DIR, "kwic_merged.csv"), index=False)

data.head(3)

dovere_raw_kwic.csv: loaded 9673 rows
potere_raw_kwic.csv: loaded 9679 rows
volere_raw_kwic.csv: loaded 9749 rows
Merged dataset size: 28500
Columns: ['Reference', 'Left', 'KWIC', 'Right', 'modal_lemma', 'src_file']


,Reference,Left,KWIC,Right,modal_lemma,src_file
0,ambientediritto.it,e modificare gli indicatori di valutazione. </...,deve risultare,"utile e, quindi, pertinente per le tre categor...",dovere,dovere_raw_kwic.csv
1,benessere.com,", l''elezione di un RLS è una facoltà concessa...",deve partecipare,– a spese del Datore di Lavoro e in orario di ...,dovere,dovere_raw_kwic.csv
2,ilgiornale.it,", nel qual caso l''avverso. </s><s> Nel caso d...",dovrei oppormi,. </s><s> Mi auguro che queste non siano solo ...,dovere,dovere_raw_kwic.csv


## Text reconstruction and infinitive normalization

In [ ]:
KW_COL_CANDS = ["KWIC", "kw", "keyword", "node", "KW", "K"]
LEFT_COL_CANDS = ["Left", "left", "L", "lc", "left_context"]
RIGHT_COL_CANDS = ["Right", "right", "R", "rc", "right_context"]
ID_COL_CANDS = ["sent_id", "sid", "id", "sentence_id", "Reference", "Ref", "ID"]

def pick_col(cols, cands):
    for c in cands:
        if c in cols:
            return c
    return None


kw_col = pick_col(list(data.columns), KW_COL_CANDS)
left_col = pick_col(list(data.columns), LEFT_COL_CANDS)
right_col = pick_col(list(data.columns), RIGHT_COL_CANDS)
id_col = pick_col(list(data.columns), ID_COL_CANDS)

print("Detected columns:")
print("KWIC:", kw_col, "| Left:", left_col, "| Right:", right_col, "| ID:", id_col)

if not kw_col or not left_col or not right_col:
    raise ValueError("Input files must contain columns for Left, KWIC, and Right context.")


def after_last_marker(text, marker="<s>"):
    parts = str(text).split(marker)
    return parts[-1]


def before_next_marker(text, marker="<s>"):
    parts = str(text).split(marker)
    return parts[0]


def strip_sentence_tags(txt):
    return re.sub(r"</?s>", "", str(txt))


def norm_spaces(txt):
    return re.sub(r"\s+", " ", str(txt)).strip()


def split_modal_and_rest(kw_str):
    """Split KWIC into modal token and following material."""
    kw_clean = norm_spaces(kw_str)

    if not kw_clean:
        return "", ""

    parts = kw_clean.split(maxsplit=1)

    if len(parts) == 1:
        return parts[0], ""

    return parts[0], parts[1]

In [ ]:
CLITICS = (
    "mi", "ti", "si", "ci", "vi", "lo", "la", "li", "le", "ne", "gli", "le",
    "m'", "t'", "s'", "c'", "v'", "l'", "gl'"
)

def _tok_keep_letters_apost(s):
    return [
        re.sub(r"^[^A-Za-zÀ-ÿ']+|[^A-Za-zÀ-ÿ']+$", "", t.lower())
        for t in str(s).split()
        if t.strip()
    ]


def _clean(tok):
    return re.sub(r"[^A-Za-zÀ-ÿ']+", "", tok.lower())


IRREG_ENCLITIC_MAP = {
    "oppormi": "opporre", "opporti": "opporre", "opporsi": "opporre",
    "opporci": "opporre", "opporvi": "opporre", "opporlo": "opporre",
    "opporla": "opporre", "opporli": "opporre", "opporle": "opporre",
    "pormi": "porre", "porti": "porre", "porsi": "porre", "porci": "porre",
    "porvi": "porre", "porlo": "porre", "porla": "porre", "porli": "porre",
    "porle": "porre",
    "trarmi": "trarre", "trarti": "trarre", "trarsi": "trarre",
    "trarci": "trarre", "trarvi": "trarre", "trarlo": "trarre",
    "trarla": "trarre", "trarli": "trarre", "trarle": "trarre",
    "condurlo": "condurre", "condurla": "condurre",
    "condurli": "condurre", "condurle": "condurre",
    "produrlo": "produrre", "produrla": "produrre",
    "tradurlo": "tradurre", "tradurla": "tradurre",
    "introdurlo": "introdurre", "introdurla": "introdurre",
    "ridurlo": "ridurre", "ridurla": "ridurre",
    "sedurlo": "sedurre", "sedurla": "sedurre",
    "dedurlo": "dedurre", "dedurla": "dedurre",
}

_RRE_BASES = {
    "por": "porre",
    "trar": "trarre",
    "condur": "condurre",
    "produr": "produrre",
    "tradur": "tradurre",
    "introdur": "introdurre",
    "ridur": "ridurre",
    "sedur": "sedurre",
    "dedur": "dedurre",
}

RRE_PATTERN = re.compile(
    r"("
    + "|".join(map(re.escape, _RRE_BASES.keys()))
    + r")("
    + "|".join(map(re.escape, CLITICS))
    + r")$"
)

BARE_TRUNC_TO_INF = {
    "far": "fare",
    "dir": "dire",
    "dar": "dare",
    "star": "stare",
    "andar": "andare",
}

def normalize_infinitive_candidate(tok):
    t = _clean(tok)

    if t in IRREG_ENCLITIC_MAP:
        return IRREG_ENCLITIC_MAP[t]

    if t in BARE_TRUNC_TO_INF:
        return BARE_TRUNC_TO_INF[t]

    if re.search(r"(are|ere|ire)$", t):
        return t

    m = re.match(r"(.+?)(ar|er|ir)(" + "|".join(map(re.escape, CLITICS)) + r")$", t)

    if m:
        stem, theme, _ = m.groups()
        return stem + theme + "e"

    m = re.match(r"(far|dir|dar|star|andar)(" + "|".join(map(re.escape, CLITICS)) + r")$", t)

    if m:
        return {
            "far": "fare",
            "dir": "dire",
            "dar": "dare",
            "star": "stare",
            "andar": "andare"
        }[m.group(1)]

    m = re.match(r"(aver|esser)(" + "|".join(map(re.escape, CLITICS)) + r")$", t)

    if m:
        return {
            "aver": "avere",
            "esser": "essere"
        }[m.group(1)]

    m = RRE_PATTERN.match(t)

    if m:
        base = m.group(1)
        return _RRE_BASES[base]

    return None


def inf_from_kw_rest(kw_rest):
    toks = _tok_keep_letters_apost(kw_rest)

    if not toks:
        return None, None

    raw = toks[0]
    norm = normalize_infinitive_candidate(raw)

    return raw, norm

## Loading the masked language model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

model.eval()

if DEVICE == "cuda" and torch.cuda.is_available():
    model.to("cuda")

print("Model loaded on:", model.device)
print("Mask token:", tokenizer.mask_token)

## Modal forms and tokenizer coverage

The model is evaluated in a closed set of modal lemmas. For each modal lemma, probabilities are obtained by summing the probabilities of its single-token surface forms.

In [ ]:
FORMS = {
    "dovere": [
        "devo", "devi", "deve", "dobbiamo", "dovete", "devono", "debbono", "dovuto",
        "dovevo", "dovevi", "doveva", "dovevamo", "dovevate", "dovevano",
        "dovetti", "dovesti", "dovette", "dové", "dovemmo", "doveste", "dovettero",
        "doverono", "dovrò", "dovrai", "dovrà", "dovremo", "dovrete", "dovranno",
        "deva", "debba", "dobbiate", "debbano", "dovessi", "dovesse", "dovessimo",
        "doveste", "dovessero", "dovrei", "dovresti", "dovrebbe", "dovremmo",
        "dovreste", "dovrebbero"
    ],
    "potere": [
        "posso", "puoi", "può", "possiamo", "potete", "possono", "potuto",
        "potevo", "potevi", "poteva", "potevamo", "potevate", "potevano",
        "potetti", "potei", "potesti", "potette", "poté", "potemmo", "poteste",
        "potettero", "poterono", "potrò", "potrai", "potrà", "potremo", "potrete",
        "potranno", "potrei", "potresti", "potrebbe", "potremmo", "potreste",
        "potrebbero", "possa", "possiate", "possano", "potessi", "potesse",
        "potessimo", "poteste", "potessero"
    ],
    "volere": [
        "voglio", "vuoi", "vuole", "vogliamo", "volete", "vogliono", "voluto",
        "volevo", "volevi", "voleva", "volevamo", "volevate", "volevano",
        "volli", "volesti", "volle", "volemmo", "voleste", "vollero",
        "vorrò", "vorrai", "vorrà", "vorremo", "vorrete", "vorranno",
        "voglia", "vogliate", "vogliano", "volessi", "volesse", "volessimo",
        "voleste", "volessero", "vorrei", "vorresti", "vorrebbe", "vorremmo",
        "vorreste", "vorrebbero"
    ]
}

SINGLE_IDS = {lemma: [] for lemma in FORMS}
MULTI_FORMS = {lemma: [] for lemma in FORMS}

for lemma, forms in FORMS.items():
    for f in forms:
        toks = tokenizer.tokenize(f)

        if len(toks) == 1:
            SINGLE_IDS[lemma].append(tokenizer.convert_tokens_to_ids(toks[0]))
        else:
            MULTI_FORMS[lemma].append((f, toks))

print("Single-token counts:", {k: len(v) for k, v in SINGLE_IDS.items()})
print("Multi-piece counts:", {k: len(v) for k, v in MULTI_FORMS.items()})

for lemma in ["dovere", "potere", "volere"]:
    print("\nMulti-piece forms for", lemma)

    for f, toks in MULTI_FORMS[lemma]:
        print(f"  {f:>12s} -> {toks}")

## Reconstructing sentences and masking the modal

In [ ]:
def rebuild_and_mask(row, mask_token):
    left_raw = row[left_col]
    right_raw = row[right_col]
    kw_raw = row[kw_col]

    left_piece = norm_spaces(strip_sentence_tags(after_last_marker(left_raw)))
    right_piece = norm_spaces(strip_sentence_tags(before_next_marker(right_raw)))

    modal_tok, kw_rest = split_modal_and_rest(kw_raw)
    kw_full = norm_spaces(f"{modal_tok} {kw_rest}".strip())

    prefix = left_piece

    if prefix and kw_full:
        prefix = prefix + " "

    sentence = norm_spaces(
        prefix + kw_full + (" " + right_piece if right_piece else "")
    )

    if not modal_tok:
        modal_start = None
        modal_end = None
        masked = sentence
        masked_modal_surface = None
    else:
        modal_start = (len(left_piece) + 1) if left_piece else 0
        modal_end = modal_start + len(modal_tok)

        masked = sentence[:modal_start] + mask_token + sentence[modal_end:]
        masked = norm_spaces(masked)

        masked_modal_surface = modal_tok

    inf_raw, inf_norm = inf_from_kw_rest(kw_rest)
    sent_id = row[id_col] if (id_col and id_col in row) else None

    return pd.Series({
        "sent_id": sent_id,
        "sentence": sentence,
        "masked": masked,
        "kwic_norm": kw_full,
        "masked_modal_surface": masked_modal_surface,
        "modal_start": modal_start,
        "modal_end": modal_end,
        "inf_raw": inf_raw,
        "inf_form": inf_norm,
    })


MASK_TOKEN = tokenizer.mask_token

sent_tbl = data.apply(
    lambda r: rebuild_and_mask(r, MASK_TOKEN),
    axis=1
)

data2 = pd.concat([data.reset_index(drop=True), sent_tbl], axis=1)

preview_cols = [
    "sent_id", "sentence", "masked", "modal_start", "modal_end",
    "kwic_norm", "masked_modal_surface", "inf_raw", "inf_form",
    "modal_lemma", "src_file"
]

display(data2[preview_cols].head(10))

data2.to_csv(
    os.path.join(OUT_DIR, "table_sentence_mask_spans.csv"),
    index=False
)

                       sent_id  \
0           ambientediritto.it   
1                benessere.com   
2                ilgiornale.it   
3          staffettaonline.com   
4                 confetra.com   
5                 blogspot.com   
6     massaggioconnettivale.it   
7         normativaitaliana.it   
8  thetuscanweddingnetwork.net   
9                    camera.it   

                                            sentence  \
0  f) gli utenti della formazione e la valutazion...   
1  Una volta eletto, il RLS deve partecipare – a ...   
2  Nel caso dei matrimoni gay non viene leso ness...   
3  È in dirittura d''arrivo il decreto interminis...   
4  In particolare essi devono provvedere ad annul...   
5  Negli scali aerei e negli snodi come le stazio...   
6  visita? sarebbe comne stampare dei biglietti d...   
7  Esso è ripartito fra i comuni rivieraschi con ...   
8  di Elia, Francesca e Jo (in ordine alfabetico....   
9  circostanza, per dire che non va necessariamen...   

        

## Sanity checks on reconstructed sentences

In [ ]:
diag = data2.copy()

sample_n = 40

show = diag.sample(n=min(sample_n, len(diag)), random_state=SEED)[
    [
        "modal_lemma", "masked_modal_surface", "kwic_norm",
        "inf_raw", "inf_form", "sentence", "masked"
    ]
]

display(show)

top_surfaces = (
    diag
    .groupby(["modal_lemma", "masked_modal_surface"])
    .size()
    .reset_index(name="n")
    .sort_values(["modal_lemma", "n"], ascending=[True, False])
)

for m in ["dovere", "potere", "volere"]:
    print("\nTop masked modal surfaces for:", m)
    print(top_surfaces[top_surfaces["modal_lemma"] == m].head(15))

fail_inf = diag["inf_form"].isna().mean()

print(f"\nInfinitive extraction failure rate: {fail_inf * 100:.2f}%")

,modal_lemma,masked_modal_surface,kwic_norm,inf_raw,inf_form,sentence,masked
9071,dovere,deve,deve venire,venire,venire,)... poi altri consigli sono di cominciare ad ...,)... poi altri consigli sono di cominciare ad ...
9222,dovere,dovrà,dovrà soddisfare,soddisfare,soddisfare,È necessario quindi stabilire gli obiettivi ch...,È necessario quindi stabilire gli obiettivi ch...
2157,dovere,dovrà,dovrà essere,essere,essere,"La parte interna della zona falcata, invece, r...","La parte interna della zona falcata, invece, r..."
19246,volere,vorrei,vorrei convincere,convincere,convincere,Soprattutto vorrei convincere qualcuno che gli...,Soprattutto [MASK] convincere qualcuno che gli...
5572,dovere,deve,deve essere,essere,essere,Se la Garanzia giovani non deve essere un pian...,Se la Garanzia giovani non [MASK] essere un pi...
18396,potere,può,può dare,dare,dare,il fatto che non si collegava a skype -.-'' la...,il fatto che non si collegava a skype -.-'' la...
19636,volere,vorrei,vorrei sostituirlo,sostituirlo,sostituire,Al momento ho un Galaxy S che è praticamente u...,Al momento ho un Galaxy S che è praticamente u...
8673,dovere,devi,devi precipitare,precipitare,precipitare,lo dira il tuo gine praticamente per sincroniz...,lo dira il tuo gine praticamente per sincroniz...
2664,dovere,deve,deve essere,essere,essere,"viene eliminato attraverso il latte, comunque ...","viene eliminato attraverso il latte, comunque ..."
15292,potere,può,può essere,essere,essere,La crescita relativamente bassa delle scorte p...,La crescita relativamente bassa delle scorte [...



Top masked modal surfaces for: dovere
   modal_lemma masked_modal_surface     n
39      dovere                 deve  2041
80      dovere             dovrebbe   879
42      dovere               devono   749
41      dovere                 devo   590
94      dovere               dovuto   519
43      dovere             dobbiamo   514
89      dovere                dovrà   452
40      dovere                 devi   389
49      dovere                dover   321
82      dovere           dovrebbero   302
71      dovere               doveva   275
79      dovere             dovranno   262
37      dovere                debba   188
62      dovere              dovesse   159
67      dovere               dovete   120

Top masked modal surfaces for: potere
    modal_lemma masked_modal_surface     n
184      potere                  può  2467
134      potere              possono  1167
168      potere             potrebbe   665
139      potere                poter   601
129      potere                poss

## Closed-set modal prediction

For each masked sentence, the model predicts probabilities over the full vocabulary. Modal-lemma probabilities are then computed by summing the probabilities of the single-token surface forms associated with each modal lemma.

In [ ]:
MASK_ID = tokenizer.mask_token_id

@torch.no_grad()
def lemma_probs_closed_fast_batch(masked_texts):
    """
    Compute closed-set probabilities for dovere, potere, and volere.

    Sentences without a mask token are skipped.
    """
    if len(masked_texts) == 0:
        return [], [], []

    enc = tokenizer(
        masked_texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    input_ids = enc["input_ids"]

    mask_positions = (input_ids == MASK_ID).nonzero(as_tuple=False)

    if mask_positions.numel() == 0:
        return [], [], []

    first_mask = {}

    for bi, pos in mask_positions.tolist():
        if bi not in first_mask:
            first_mask[bi] = pos

    valid_idx = sorted(first_mask.keys())
    mask_pos_list = torch.tensor(
        [first_mask[i] for i in valid_idx],
        device=model.device
    )

    enc_valid = {
        k: v[valid_idx]
        for k, v in enc.items()
    }

    logits = model(**enc_valid).logits

    Bv = logits.shape[0]
    row_idx = torch.arange(Bv, device=model.device)
    mask_logits = logits[row_idx, mask_pos_list, :]

    probs = F.softmax(mask_logits, dim=-1)

    raw_do = (
        probs[:, SINGLE_IDS["dovere"]].sum(dim=1)
        if len(SINGLE_IDS["dovere"])
        else torch.zeros(Bv, device=model.device)
    )

    raw_po = (
        probs[:, SINGLE_IDS["potere"]].sum(dim=1)
        if len(SINGLE_IDS["potere"])
        else torch.zeros(Bv, device=model.device)
    )

    raw_vo = (
        probs[:, SINGLE_IDS["volere"]].sum(dim=1)
        if len(SINGLE_IDS["volere"])
        else torch.zeros(Bv, device=model.device)
    )

    raw_stack = torch.stack([raw_do, raw_po, raw_vo], dim=1)

    total_modal = raw_stack.sum(dim=1)
    total_modal_safe = torch.where(
        total_modal > 0,
        total_modal,
        torch.ones_like(total_modal)
    )

    closed = raw_stack / total_modal_safe.unsqueeze(1)
    p_other = (1.0 - total_modal).clamp_min(0.0)

    closed_list = []

    for i in range(Bv):
        closed_list.append({
            "dovere": float(closed[i, 0].item()),
            "potere": float(closed[i, 1].item()),
            "volere": float(closed[i, 2].item())
        })

    p_other_list = [float(x.item()) for x in p_other]

    return closed_list, p_other_list, valid_idx

In [ ]:
BATCH_SIZE = 32

rows = data2.copy()
preds = []
skipped_no_mask = 0

for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="Scoring sentences"):
    end = min(start + BATCH_SIZE, len(rows))
    batch = rows.iloc[start:end].copy()

    masked_texts = batch["masked"].tolist()
    closed_list, p_other_list, valid_idx = lemma_probs_closed_fast_batch(masked_texts)

    skipped_no_mask += len(masked_texts) - len(valid_idx)

    for local_i, closed, p_other in zip(valid_idx, closed_list, p_other_list):
        row = batch.iloc[local_i]
        modal_hat = max(closed.items(), key=lambda kv: kv[1])[0]

        preds.append({
            "sent_id": row.get("sent_id"),
            "sentence": row["sentence"],
            "masked": row["masked"],
            "masked_modal_surface": row.get("masked_modal_surface"),
            "modal_start": row.get("modal_start"),
            "modal_end": row.get("modal_end"),
            "modal_gold": row["modal_lemma"],
            "kwic": row["kwic_norm"],
            "inf_raw": row.get("inf_raw"),
            "inf_form": row.get("inf_form"),
            "P_do_model": closed.get("dovere", 0.0),
            "P_po_model": closed.get("potere", 0.0),
            "P_vo_model": closed.get("volere", 0.0),
            "P_other": p_other,
            "pred_modal": modal_hat,
            "src_file": row["src_file"],
        })

pred_df = pd.DataFrame(preds)

pred_df.to_csv(
    os.path.join(OUT_DIR, "predictions_rows.csv"),
    index=False
)

print(f"Total input rows: {len(rows)}")
print(f"Rows scored: {len(pred_df)}")
print(f"Rows skipped because no mask was found: {skipped_no_mask}")

pred_df.head(5)

Scoring sentences (batched 32):   0%|          | 0/891 [00:00<?, ?it/s]

Total rows input: 28500
Rows scored (had mask): 28500
Rows skipped (no mask found after rebuild): 0


,sent_id,sentence,masked,masked_modal_surface,modal_start,modal_end,modal_gold,kwic,inf_raw,inf_form,P_do_model,P_po_model,P_vo_model,P_other,pred_modal,src_file
0,ambientediritto.it,f) gli utenti della formazione e la valutazion...,f) gli utenti della formazione e la valutazion...,deve,74,78,dovere,deve risultare,risultare,risultare,0.326528,0.673253,0.000220,0.029628,potere,dovere_raw_kwic.csv
1,benessere.com,"Una volta eletto, il RLS deve partecipare – a ...","Una volta eletto, il RLS [MASK] partecipare – ...",deve,25,29,dovere,deve partecipare,partecipare,partecipare,0.099606,0.900038,0.000356,0.001314,potere,dovere_raw_kwic.csv
2,ilgiornale.it,Nel caso dei matrimoni gay non viene leso ness...,Nel caso dei matrimoni gay non viene leso ness...,dovrei,92,98,dovere,dovrei oppormi,oppormi,opporre,0.945311,0.040720,0.013969,0.026626,dovere,dovere_raw_kwic.csv
3,staffettaonline.com,È in dirittura d''arrivo il decreto interminis...,È in dirittura d''arrivo il decreto interminis...,deve,87,91,dovere,deve individuare,individuare,individuare,0.820554,0.071600,0.107846,0.352217,dovere,dovere_raw_kwic.csv
4,confetra.com,In particolare essi devono provvedere ad annul...,In particolare essi [MASK] provvedere ad annul...,devono,20,26,dovere,devono provvedere,provvedere,provvedere,0.923038,0.074152,0.002810,0.012466,dovere,dovere_raw_kwic.csv


## Sentence-level evaluation

In [ ]:
labels = ["dovere","potere","volere"]

gold = pred_df["modal_gold"].tolist()
pred = pred_df["pred_modal"].tolist()

acc = float(np.mean([g==p for g,p in zip(gold,pred)]))

def f1_macro(y_true, y_pred, labels):
    f1s = []
    for lab in labels:
        tp = sum((yt==lab and yp==lab) for yt,yp in zip(y_true,y_pred))
        fp = sum((yt!=lab and yp==lab) for yt,yp in zip(y_true,y_pred))
        fn = sum((yt==lab and yp!=lab) for yt,yp in zip(y_true,y_pred))
        prec = tp / (tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp / (tp+fn) if (tp+fn)>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        f1s.append(f1)
    return float(np.mean(f1s))

f1 = f1_macro(gold, pred, labels)

cm = pd.crosstab(pd.Series(gold, name="gold"),
                 pd.Series(pred, name="pred"),
                 rownames=["gold"], colnames=["pred"],
                 dropna=False).reindex(index=labels, columns=labels, fill_value=0)

print(f"Accuracy: {acc:.4f} | Macro-F1: {f1:.4f}")
print(cm)

# per-modal (per-class) accuracy and F1

per_modal_rows = []

for lab in labels:
    tp = sum((g==lab and p==lab) for g,p in zip(gold,pred))
    tn = sum((g!=lab and p!=lab) for g,p in zip(gold,pred))
    fp = sum((g!=lab and p==lab) for g,p in zip(gold,pred))
    fn = sum((g==lab and p!=lab) for g,p in zip(gold,pred))

    acc_lab = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_lab = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0

    per_modal_rows.append({
        "modal": lab,
        "accuracy": acc_lab,
        "f1": f1_lab
    })

per_modal_df = pd.DataFrame(per_modal_rows).set_index("modal")
print("\nPer-modal accuracy and F1:")
print(per_modal_df)

# 1) Metrics summary
metrics_df = pd.DataFrame([{
    "model": MODEL_NAME,
    "rows_scored": len(pred_df),
    "accuracy": acc,
    "macro_f1": f1
}])

metrics_path = os.path.join(OUT_DIR, "task2_eval_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

# per-modal metrics CSV
per_modal_path = os.path.join(OUT_DIR, "task2_per_modal_metrics.csv")
per_modal_df.to_csv(per_modal_path)

# 2) Confusion matrix
cm_path = os.path.join(OUT_DIR, "task2_confusion_matrix.csv")
cm.to_csv(cm_path)

print("\nSaved CSV files:")
print(" -", metrics_path)
print(" -", per_modal_path)
print(" -", cm_path)

Accuracy: 0.7275 | Macro-F1: 0.7271
pred    dovere  potere  volere
gold                          
dovere    6536    2160     804
potere    1127    7893     480
volere    1513    1681    6306

Per-modal accuracy and F1:
        accuracy        f1
modal                     
dovere  0.803368  0.699936
potere  0.808842  0.743430
volere  0.842877  0.737975

Saved CSV files:
 - out_task2_modal_mask/task2_eval_metrics.csv
 - out_task2_modal_mask/task2_per_modal_metrics.csv
 - out_task2_modal_mask/task2_confusion_matrix.csv


## Loading covarying collexeme gold data

This section loads the covarying collexeme output and builds a partial gold ranking of modal preferences for each infinitive.

In [ ]:
def smart_read_covar(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]

    required = {"SLOT1","SLOT2","ASSOC","COLL.STR.LOGL"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing required columns in covar file. Need {required}, got {set(df.columns)}")

    df["SLOT1"] = df["SLOT1"].astype(str).str.strip().str.lower()
    df["SLOT2"] = df["SLOT2"].astype(str).str.strip().str.lower()
    df["ASSOC"] = df["ASSOC"].astype(str).str.strip().str.lower()
    df["COLL.STR.LOGL"] = pd.to_numeric(df["COLL.STR.LOGL"], errors="coerce")

    sign = df["ASSOC"].map({"attr": 1.0, "rep": -1.0})
    df["S_signed"] = df["COLL.STR.LOGL"] * sign

    df = df[df["SLOT1"].isin(["dovere","potere","volere"])].copy()
    df = df.dropna(subset=["S_signed","SLOT2","SLOT1"])
    return df

cov = smart_read_covar(COVAR_FILE)

cov = (cov.sort_values("S_signed", key=lambda s: s.abs(), ascending=False)
          .drop_duplicates(subset=["SLOT2","SLOT1"], keep="first"))

# Build partial gold per infinitive
gold_rows = []
for inf, sub in cov.groupby("SLOT2"):
    scores_present = {row["SLOT1"]: float(row["S_signed"]) for _, row in sub.iterrows()}
    rank_present = [m for m,_ in sorted(scores_present.items(), key=lambda kv: -kv[1])]
    gold_rows.append({
        "inf_lemma": inf,
        "rank_present": rank_present,
        "scores_present": scores_present,   # dict, used for partial pairwise scoring
        "n_present": len(scores_present)
    })

gold_partial = pd.DataFrame(gold_rows)
gold_partial.to_csv(os.path.join(OUT_DIR, "gold_partial_present_only.csv"), index=False)

print("Gold infinitives:", len(gold_partial))
print(gold_partial["n_present"].value_counts().sort_index())
gold_partial.head(10)


Gold infinitives: 1343
n_present
1    381
2    267
3    695
Name: count, dtype: int64


,inf_lemma,rank_present,scores_present,n_present
0,abbandonare,"[dovere, volere, potere]","{'potere': -8299.95037, 'dovere': 5372.59708, ...",3
1,abbassare,"[dovere, potere, volere]","{'dovere': 622.96435, 'volere': -190.1399, 'po...",3
2,abbattere,"[volere, dovere, potere]","{'volere': 678.82241, 'potere': -483.08808, 'd...",3
3,abbinare,"[potere, volere, dovere]","{'dovere': -1151.08738, 'potere': 635.38886, '...",3
4,abbracciare,"[volere, potere, dovere]","{'volere': 2953.73245, 'dovere': -1433.26161, ...",3
5,abilitare,"[dovere, potere]","{'dovere': 5254.34802, 'potere': -701.70716}",2
6,abitare,"[volere, potere, dovere]","{'volere': 341.29892, 'dovere': -94.91625, 'po...",3
7,abituare,"[dovere, volere]","{'dovere': 9386.23581, 'volere': -1807.98257}",2
8,abolire,"[volere, dovere]","{'volere': 5791.01688, 'dovere': -760.62813}",2
9,abortire,[volere],{'volere': 3470.95943},1


## Aggregating model probabilities by infinitive

In [ ]:
MIN_OCC = 3

agg = (
    pred_df.dropna(subset=["inf_form"])
           .assign(inf_lemma=lambda d: d["inf_form"].str.lower())
)

counts = agg["inf_lemma"].value_counts()
keep_lemmas = counts[counts >= MIN_OCC].index
agg = agg[agg["inf_lemma"].isin(keep_lemmas)]

per_inf = (
    agg.groupby("inf_lemma", as_index=False)[["P_do_model","P_po_model","P_vo_model"]]
       .mean()
)

def rank3_from_probs(row):
    pairs = [("dovere", row["P_do_model"]),
             ("potere", row["P_po_model"]),
             ("volere", row["P_vo_model"])]
    return [m for m,_ in sorted(pairs, key=lambda x: -x[1])]

per_inf["rank_model"] = per_inf.apply(rank3_from_probs, axis=1)

per_inf.to_csv(os.path.join(OUT_DIR, "model_per_inf_probs_and_rank.csv"), index=False)
per_inf.head(10)

,inf_lemma,P_do_model,P_po_model,P_vo_model,rank_model
0,abbandonare,0.661402,0.103006,0.235592,"[dovere, volere, potere]"
1,abbassare,0.587574,0.356015,0.056411,"[dovere, potere, volere]"
2,abbattere,0.268737,0.227183,0.504080,"[volere, dovere, potere]"
3,abbinare,0.245949,0.354232,0.399819,"[volere, potere, dovere]"
4,abbracciare,0.258048,0.314624,0.427328,"[volere, potere, dovere]"
5,abilitare,0.456844,0.361207,0.181948,"[dovere, potere, volere]"
6,abituare,0.757429,0.146018,0.096553,"[dovere, potere, volere]"
7,abolire,0.142078,0.052386,0.805535,"[volere, dovere, potere]"
8,accadere,0.281099,0.713377,0.005524,"[potere, dovere, volere]"
9,accantonare,0.519616,0.403578,0.076805,"[dovere, potere, volere]"


## Inspecting top modal predictions by infinitive

In [ ]:
in_path = os.path.join(OUT_DIR, "model_per_inf_probs_and_rank.csv")
per_inf = pd.read_csv(in_path)

rank_col = "rank_model"

def get_top_modal(rank_val):
    """Extract the top-ranked modal from the stored model ranking."""
    if isinstance(rank_val, list) and len(rank_val) > 0:
        return str(rank_val[0]).strip().lower()

    s = str(rank_val).strip().lower()
    if s.startswith("[") and s.endswith("]"):
        parts = s.replace('"', "'").split("'")
        toks = [p.strip() for p in parts if p.strip() and p.strip() not in [",", "[", "]"]]
        return toks[0] if toks else None

    for sep in [">", ",", ";", "|", " "]:
        if sep in s:
            if sep == ">":
                left = s.split(">")[0].strip()
                return left
            if sep in [",", ";", "|"]:
                left = s.split(sep)[0].strip()
                return left

    return s if s in {"dovere", "potere", "volere"} else None

per_inf["top_modal"] = per_inf[rank_col].apply(get_top_modal)

#Count infinitives by top predicted modal
labels = ["dovere", "potere", "volere"]

top_counts = (
    per_inf["top_modal"]
    .value_counts()
    .reindex(labels, fill_value=0)
    .reset_index()
)
top_counts.columns = ["top_modal", "n_infinitives"]

print("\nNumber of infinitives by top predicted modal:")
print(top_counts)

counts_path = os.path.join(OUT_DIR, "task2_top_modal_counts.csv")
top_counts.to_csv(counts_path, index=False)


#Top 15 infinitives per modal (top-ranked + highest prob for that modal)
base_cols = ["inf_lemma", "P_do_model", "P_po_model", "P_vo_model", "rank_model"]

prob_col = {"dovere": "P_do_model", "potere": "P_po_model", "volere": "P_vo_model"}

top15_tables = {}

for m in labels:
    df_m = (
        per_inf[per_inf["top_modal"] == m]
        .sort_values(prob_col[m], ascending=False)
        .loc[:, base_cols]
        .head(15)
        .reset_index(drop=True)
    )

    top15_tables[m] = df_m

    print(f"\nTop 15 infinitives with top prediction = {m} (highest P_{m}):")
    print(df_m)

    out_path = os.path.join(OUT_DIR, f"task2_top15_{m}_infinitives.csv")
    df_m.to_csv(out_path, index=False)

print("\nSaved CSV files:")
print(" -", counts_path)
for m in labels:
    print(" -", os.path.join(OUT_DIR, f"task2_top15_{m}_infinitives.csv"))


Number of infinitives by top predicted modal:
  top_modal  n_infinitives
0    dovere            274
1    potere            467
2    volere            231

Top 15 infinitives with top prediction = dovere (highest P_dovere):
        inf_lemma  P_do_model  P_po_model  P_vo_model  \
0       pervenire    0.966522    0.033020    0.000458   
1        attenere    0.963795    0.020605    0.015600   
2        sforzare    0.946671    0.037979    0.015350   
3        sbrigare    0.928398    0.040720    0.030882   
4        scontare    0.926120    0.041328    0.032553   
5        spettare    0.921330    0.077420    0.001250   
6      conformare    0.914065    0.046491    0.039444   
7     preoccupare    0.880346    0.094010    0.025643   
8        redigere    0.865686    0.116572    0.017742   
9         tendere    0.854620    0.109376    0.036004   
10     rassegnare    0.848162    0.116985    0.034853   
11  corrispondere    0.838906    0.149843    0.011251   
12         sudare    0.825767    0.

## Merging model rankings with partial gold rankings

In [ ]:
eval_df = per_inf.merge(gold_partial, on="inf_lemma", how="left")

coverage = eval_df["rank_present"].notna().mean()
n_total = len(eval_df)
n_have_gold = int(eval_df["rank_present"].notna().sum())
n_missing_gold = n_total - n_have_gold

print(f"Per-infinitive merge coverage: {coverage*100:.1f}%")
print(f"Total model infinitives (>=MIN_OCC): {n_total}")
print(f"With gold: {n_have_gold}")
print(f"Missing gold: {n_missing_gold}")

# Show examples of missing gold
missing_examples = eval_df[eval_df["rank_present"].isna()].head(30)[
    ["inf_lemma","rank_model","P_do_model","P_po_model","P_vo_model"]
]
print("\nExamples where model has an infinitive but gold is missing (first 30):")
display(missing_examples)

# Show examples of matched
matched_examples = eval_df[eval_df["rank_present"].notna()].head(30)[
    ["inf_lemma","rank_model","rank_present","n_present","P_do_model","P_po_model","P_vo_model"]
]
print("\nExamples where model and gold both exist (first 30):")
display(matched_examples)

eval_df.to_csv(os.path.join(OUT_DIR, "eval_merged_model_gold_partial.csv"), index=False)


Per-infinitive merge coverage: 96.1%
Total model infinitives (>=MIN_OCC): 972
With gold: 934
Missing gold: 38

Examples where model has an infinitive but gold is missing (first 30):


,inf_lemma,rank_model,P_do_model,P_po_model,P_vo_model
19,accorciare,"[potere, dovere, volere]",0.346601,0.385998,0.267401
74,appacificare,"[volere, dovere, potere]",0.122204,0.100110,0.777686
78,appastare,"[volere, dovere, potere]",0.375400,0.120269,0.504331
93,arbitrare,"[dovere, volere, potere]",0.622746,0.098092,0.279162
100,arretrare,"[dovere, volere, potere]",0.660339,0.080382,0.259280
141,avvolgere,"[volere, dovere, potere]",0.192802,0.142557,0.664641
157,calcare,"[potere, dovere, volere]",0.269954,0.515634,0.214412
235,consacrare,"[volere, dovere, potere]",0.242834,0.188509,0.568657
261,coronare,"[potere, volere, dovere]",0.092342,0.489734,0.417924
298,digiunare,"[dovere, potere, volere]",0.542512,0.425224,0.032264



Examples where model and gold both exist (first 30):


,inf_lemma,rank_model,rank_present,n_present,P_do_model,P_po_model,P_vo_model
0,abbandonare,"[dovere, volere, potere]","[dovere, volere, potere]",3.0,0.661402,0.103006,0.235592
1,abbassare,"[dovere, potere, volere]","[dovere, potere, volere]",3.0,0.587574,0.356015,0.056411
2,abbattere,"[volere, dovere, potere]","[volere, dovere, potere]",3.0,0.268737,0.227183,0.504080
3,abbinare,"[volere, potere, dovere]","[potere, volere, dovere]",3.0,0.245949,0.354232,0.399819
4,abbracciare,"[volere, potere, dovere]","[volere, potere, dovere]",3.0,0.258048,0.314624,0.427328
5,abilitare,"[dovere, potere, volere]","[dovere, potere]",2.0,0.456844,0.361207,0.181948
6,abituare,"[dovere, potere, volere]","[dovere, volere]",2.0,0.757429,0.146018,0.096553
7,abolire,"[volere, dovere, potere]","[volere, dovere]",2.0,0.142078,0.052386,0.805535
8,accadere,"[potere, dovere, volere]","[potere, dovere]",2.0,0.281099,0.713377,0.005524
9,accantonare,"[dovere, potere, volere]",[dovere],1.0,0.519616,0.403578,0.076805


In [ ]:
rank_table = eval_df[eval_df["rank_present"].notna()][["inf_lemma","rank_model","rank_present","n_present"]].copy()
rank_table.to_csv(os.path.join(OUT_DIR, "rank_table_model_vs_gold_present_only.csv"), index=False)

rank_table.head(20)

,inf_lemma,rank_model,rank_present,n_present
0,abbandonare,"[dovere, volere, potere]","[dovere, volere, potere]",3.0
1,abbassare,"[dovere, potere, volere]","[dovere, potere, volere]",3.0
2,abbattere,"[volere, dovere, potere]","[volere, dovere, potere]",3.0
3,abbinare,"[volere, potere, dovere]","[potere, volere, dovere]",3.0
4,abbracciare,"[volere, potere, dovere]","[volere, potere, dovere]",3.0
5,abilitare,"[dovere, potere, volere]","[dovere, potere]",2.0
6,abituare,"[dovere, potere, volere]","[dovere, volere]",2.0
7,abolire,"[volere, dovere, potere]","[volere, dovere]",2.0
8,accadere,"[potere, dovere, volere]","[potere, dovere]",2.0
9,accantonare,"[dovere, potere, volere]",[dovere],1.0


## Pairwise comparison with partial gold rankings

The gold data does not necessarily provide a complete ranking over all three modals for every infinitive. Therefore, evaluation is performed using pairwise constraints defined by the available gold associations.

In [ ]:
MODALS = ["dovere", "potere", "volere"]
pairs3 = [("dovere","potere"), ("dovere","volere"), ("potere","volere")]

def gold_prefers(x, y, present_scores):
    """
    Returns:
      True  if gold asserts x > y
      False if gold asserts y > x
      None  if gold does not define the relation (missing–missing or exact tie)
    """
    x_in = x in present_scores
    y_in = y in present_scores

    if x_in and y_in:
        sx, sy = present_scores[x], present_scores[y]
        if sx == sy:
            return None
        return sx > sy

    if x_in and (not y_in):
        return True      # present beats missing
    if (not x_in) and y_in:
        return False     # missing loses to present

    return None          # both missing -> no constraint

def model_prefers(x, y, rank_model):
    pos = {m:i for i,m in enumerate(rank_model)}
    return pos[x] < pos[y]

def pairwise_accuracy_partial(rank_model, present_scores):
    correct = 0
    scorable = 0
    for x, y in pairs3:
        g = gold_prefers(x, y, present_scores)
        if g is None:
            continue
        scorable += 1
        m = model_prefers(x, y, rank_model)
        correct += int(m == g)
    acc = correct / scorable if scorable > 0 else np.nan
    return acc, scorable

tmp = eval_df.dropna(subset=["rank_present"]).copy()

res = tmp.apply(lambda r: pairwise_accuracy_partial(r["rank_model"], r["scores_present"]), axis=1)
tmp["pairwise_acc_partial"] = [a for a, n in res]
tmp["n_scorable_pairs"] = [n for a, n in res]

print("Pairwise order accuracy (partial constraints):")
print("Mean  :", float(tmp["pairwise_acc_partial"].mean(skipna=True)))
print("Median:", float(tmp["pairwise_acc_partial"].median(skipna=True)))

print("\nScorable pair-count distribution:")
print(tmp["n_scorable_pairs"].value_counts().sort_index())

print("\nBreakdown by n_present (gold):")
print(tmp.groupby("n_present")["pairwise_acc_partial"].agg(["count","mean","median"]))

tmp.to_csv(os.path.join(OUT_DIR, "pairwise_accuracy_partial_gold.csv"), index=False)

tmp.head(20)[["inf_lemma","n_present","rank_model","rank_present","pairwise_acc_partial","n_scorable_pairs"]]


Pairwise order accuracy (partial constraints):
Mean  : 0.7776588151320485
Median: 0.6666666666666666

Scorable pair-count distribution (should be 2 for n_present=1, else 3):
n_scorable_pairs
2    112
3    822
Name: count, dtype: int64

Breakdown by n_present (gold):
           count      mean    median
n_present                           
1.0          112  0.875000  1.000000
2.0          178  0.818352  1.000000
3.0          644  0.749482  0.666667


,inf_lemma,n_present,rank_model,rank_present,pairwise_acc_partial,n_scorable_pairs
0,abbandonare,3.0,"[dovere, volere, potere]","[dovere, volere, potere]",1.000000,3
1,abbassare,3.0,"[dovere, potere, volere]","[dovere, potere, volere]",1.000000,3
2,abbattere,3.0,"[volere, dovere, potere]","[volere, dovere, potere]",1.000000,3
3,abbinare,3.0,"[volere, potere, dovere]","[potere, volere, dovere]",0.666667,3
4,abbracciare,3.0,"[volere, potere, dovere]","[volere, potere, dovere]",1.000000,3
5,abilitare,2.0,"[dovere, potere, volere]","[dovere, potere]",1.000000,3
6,abituare,2.0,"[dovere, potere, volere]","[dovere, volere]",0.666667,3
7,abolire,2.0,"[volere, dovere, potere]","[volere, dovere]",1.000000,3
8,accadere,2.0,"[potere, dovere, volere]","[potere, dovere]",1.000000,3
9,accantonare,1.0,"[dovere, potere, volere]",[dovere],1.000000,2
